In [0]:
from pyspark.sql import functions as F

orders_data = [
    (1, "Laptop",   "Electronics", 1000.0, 1, 10),
    (2, "Mouse",    "Electronics",   25.0, 2,  0),
    (3, "Notebook", "Stationery",     5.0, 5, 20),
    (4, "Desk",     "Furniture",    200.0, 1,  5),
    (5, "Pen",      "Stationery",     2.0, 10, 0),
]

columns = ["order_id", "product", "category", "price", "quantity", "discount_pct"]

df_orders = spark.createDataFrame(orders_data, columns)
display(df_orders)

order_id,product,category,price,quantity,discount_pct
1,Laptop,Electronics,1000.0,1,10
2,Mouse,Electronics,25.0,2,0
3,Notebook,Stationery,5.0,5,20
4,Desk,Furniture,200.0,1,5
5,Pen,Stationery,2.0,10,0


In [0]:
df_calculated = (
    df_orders
    .withColumn(
        "gross_amount",
        F.col("price") * F.col("quantity")
    )
    .withColumn(
        "final_amount",
        F.col("gross_amount") * (1 - F.col("discount_pct") / 100.0)
    )
    .filter(F.col("final_amount") >= 20)
    .select("order_id", "product", "category", "gross_amount", "discount_pct", "final_amount")
)

display(df_calculated)

order_id,product,category,gross_amount,discount_pct,final_amount
1,Laptop,Electronics,1000.0,10,900.0
2,Mouse,Electronics,50.0,0,50.0
3,Notebook,Stationery,25.0,20,20.0
4,Desk,Furniture,200.0,5,190.0
5,Pen,Stationery,20.0,0,20.0


In [0]:
df_summary = (
    df_calculated
    .groupBy("category")
    .agg(
        F.count("order_id").alias("orders_count"),
        F.round(F.sum("final_amount"), 2).alias("total_sales")
    )
    .orderBy(F.col("total_sales").desc())
)

display(df_summary)

category,orders_count,total_sales
Electronics,2,950.0
Furniture,1,190.0
Stationery,2,40.0


In [0]:
df_summary.write.mode("overwrite").saveAsTable("sales_summary_by_category")